# 1. Controle da execução

Execute esta célula primeiro. Ela apenas define o plano; não baixa arquivos nem executa validações.

- `DATASET_TO_RUN`: dataset que será analisado.
- `RUN_DATASET_ACQUISITION`: permite baixar ou retomar arquivos ausentes.
- `FORCE_DATASET_DOWNLOAD`: baixa novamente mesmo que pareça completo.
- `FORCE_DOCKER_BUILD`: reconstrói a imagem Docker à força.
- `RUN_METADATA_VALIDATION`: executa ou reutiliza a validação técnica.
- `RUN_QUALITY_VALIDATION`: executa ou reutiliza a validação de qualidade.
- `FORCE_RERUN`: ignora resultados anteriores e valida novamente.
- `RUN_EXECUTION_REPORT`: gera um resumo humano da execução em `harness/artifacts/reports/`.
- `RUN_ARCHITECTURE_UPDATE`: tenta publicar candidatos Archify existentes; não cria um diagrama por conta própria.

Com `FORCE_* = False`, o notebook reutiliza o que já estiver válido. O relatório fica habilitado por padrão; a atualização Archify fica desabilitada até existir um candidato.

In [25]:
DATASET_TO_RUN = 'minds-libras-raw'
RUN_DATASET_ACQUISITION = True
FORCE_DATASET_DOWNLOAD = False
FORCE_DOCKER_BUILD = False
RUN_METADATA_VALIDATION = True
RUN_QUALITY_VALIDATION = True
FORCE_RERUN = False
RUN_EXECUTION_REPORT = True
RUN_ARCHITECTURE_UPDATE = True

print({
    'dataset': DATASET_TO_RUN,
    'aquisição_ou_retomada_habilitada': RUN_DATASET_ACQUISITION,
    'forçar_download': FORCE_DATASET_DOWNLOAD,
    'forçar_build_docker': FORCE_DOCKER_BUILD,
    'executar_validação_técnica': RUN_METADATA_VALIDATION,
    'executar_validação_de_qualidade': RUN_QUALITY_VALIDATION,
    'forçar_nova_validação': FORCE_RERUN,
    'gerar_relatorio_da_execução': RUN_EXECUTION_REPORT,
    'atualizar_arquitetura': RUN_ARCHITECTURE_UPDATE,
})


{'dataset': 'minds-libras-raw', 'aquisição_ou_retomada_habilitada': True, 'forçar_download': False, 'forçar_build_docker': False, 'executar_validação_técnica': True, 'executar_validação_de_qualidade': True, 'forçar_nova_validação': False, 'gerar_relatorio_da_execução': True, 'atualizar_arquitetura': True}


# 2. Preparação, aquisição e Docker

Esta seção é o ponto único de execução do fluxo. Ela verifica os pré-requisitos disponíveis, baixa ou retoma o dataset selecionado e constrói a imagem de processamento quando necessário. Os comandos são executados pelo próprio notebook; não é necessário digitá-los no terminal.

O notebook não instala Docker Desktop nem cria credenciais. Esses são pré-requisitos da máquina. O token Hugging Face é lido do ambiente ou de `app/.env` e nunca é exibido. Para repetir deliberadamente o download, altere `FORCE_DATASET_DOWNLOAD` para `True`; para reconstruir a imagem, use `FORCE_DOCKER_BUILD = True`.

In [26]:
import json
from datetime import datetime
import os
import shutil
import subprocess
from pathlib import Path

ACQUISITION_DATASET = DATASET_TO_RUN
DOCKER_IMAGE = 'libras-video-processing:0.1.0'
COMPOSE_FILE = Path.cwd()
for candidate in (COMPOSE_FILE, *COMPOSE_FILE.parents):
    if (candidate / 'pyproject.toml').is_file() and (candidate / 'data').is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Não foi possível localizar a raiz de app/.')

COMPOSE_FILE = PROJECT_ROOT / 'docker' / 'video-processing' / 'compose.yaml'
DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw' / ACQUISITION_DATASET
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / f'{ACQUISITION_DATASET}.json'

def run_notebook_command(command: list[str], *, cwd: Path = PROJECT_ROOT, label: str) -> subprocess.CompletedProcess:
    print(f'Executando: {label}')
    completed = subprocess.run(
        command, cwd=cwd, shell=False, capture_output=True,
        text=True, encoding='utf-8', errors='replace', check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f'{label} falhou com código {completed.returncode}.')
    return completed

uv_executable = shutil.which('uv')
docker_executable = shutil.which('docker')
if uv_executable is None:
    raise RuntimeError('uv não foi encontrado. Instale o uv antes de abrir o Jupyter.')
if docker_executable is None:
    raise RuntimeError('Docker não foi encontrado. Instale/inicie o Docker Desktop antes de abrir o Jupyter.')
if not COMPOSE_FILE.is_file():
    raise FileNotFoundError(f'Compose não encontrado: {COMPOSE_FILE}')
if not (os.environ.get('HF_TOKEN') or (PROJECT_ROOT / '.env').is_file()):
    raise RuntimeError('Configure HF_TOKEN em app/.env ou no ambiente antes de executar a aquisição.')
run_notebook_command([docker_executable, 'info'], label='verificação do Docker Desktop')
run_notebook_command([uv_executable, 'sync'], label='sincronização do ambiente Python')

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
expected_video_files = {
    Path(remote_path).name
    for remote_path in manifest.get('files', {})
    if str(remote_path).startswith('videos/')
}
expected_video_sizes = {
    Path(remote_path).name: details.get('expected_size_bytes')
    for remote_path, details in manifest.get('files', {}).items()
    if str(remote_path).startswith('videos/') and isinstance(details, dict)
}
videos_root = DATASET_ROOT / 'videos'
actual_video_files = {path.name for path in videos_root.glob('*.mp4')} if videos_root.is_dir() else set()
missing_video_files = expected_video_files - actual_video_files
size_mismatched_files = {
    name for name in expected_video_files & actual_video_files
    if expected_video_sizes.get(name) is not None
    and (videos_root / name).stat().st_size != expected_video_sizes[name]
}
annotations_path = DATASET_ROOT / 'annotations.csv'
download_needed = not annotations_path.is_file() or bool(missing_video_files or size_mismatched_files)
if not RUN_DATASET_ACQUISITION and download_needed:
    raise RuntimeError('A aquisição está desabilitada, mas o dataset está incompleto. Altere RUN_DATASET_ACQUISITION para True.')
if RUN_DATASET_ACQUISITION and (FORCE_DATASET_DOWNLOAD or download_needed):
    run_notebook_command(
        [uv_executable, 'run', 'libras', 'data', 'download', '--dataset', ACQUISITION_DATASET],
        label=f'aquisição/retomada de {ACQUISITION_DATASET}',
    )
elif RUN_DATASET_ACQUISITION:
    print(f'Dataset já completo segundo o manifesto: {ACQUISITION_DATASET}')
else:
    print(f'Aquisição desabilitada; dataset existente será apenas validado: {ACQUISITION_DATASET}')

image_check = subprocess.run(
    [docker_executable, 'image', 'inspect', DOCKER_IMAGE],
    cwd=PROJECT_ROOT, shell=False, capture_output=True, text=True,
)
image_is_stale = False
if image_check.returncode == 0:
    try:
        image_created = json.loads(image_check.stdout)[0].get('Created', '')
        image_created_timestamp = datetime.fromisoformat(image_created.replace('Z', '+00:00')).timestamp()
        source_files = [PROJECT_ROOT / name for name in ('pyproject.toml', 'uv.lock', 'docker/video-processing/Dockerfile', 'docker/video-processing/compose.yaml')]
        source_files.extend((PROJECT_ROOT / 'dataset').rglob('*.py'))
        latest_source_timestamp = max(path.stat().st_mtime for path in source_files if path.is_file())
        image_is_stale = image_created_timestamp < latest_source_timestamp
    except (KeyError, TypeError, ValueError, OSError):
        image_is_stale = True
if FORCE_DOCKER_BUILD or image_check.returncode != 0 or image_is_stale:
    run_notebook_command(
        [docker_executable, 'compose', '-f', str(COMPOSE_FILE), 'build', 'video-processing'],
        label='construção da imagem Docker de processamento',
    )
else:
    print(f'Imagem Docker reutilizada: {DOCKER_IMAGE}')

actual_video_files = {path.name for path in videos_root.glob('*.mp4')} if videos_root.is_dir() else set()
print({
    'dataset': ACQUISITION_DATASET,
    'registros_esperados_no_manifesto': len(expected_video_files),
    'arquivos_mp4_encontrados': len(actual_video_files),
    'annotations_csv': annotations_path.is_file(),
    'arquivos_ausentes': len(expected_video_files - actual_video_files),
    'arquivos_com_tamanho_diferente': len(size_mismatched_files),
})


Executando: verificação do Docker Desktop
Client:
 Version:    29.9.2
 Context:    desktop-linux
 Debug Mode: true
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.152.0
    Path:     C:\Program Files\Docker\cli-plugins\docker-agent.exe
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.33.0
    Path:     C:\Program Files\Docker\cli-plugins\docker-ai.exe
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.38.1
    Path:     C:\Program Files\Docker\cli-plugins\docker-buildx.exe
  compose: Docker Compose (Docker Inc.)
    Version:  v5.6.1
    Path:     C:\Program Files\Docker\cli-plugins\docker-compose.exe
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.49.0
    Path:     C:\Program Files\Docker\cli-plugins\docker-debug.exe
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  4.96.3
    Path:     C:\Program Files\Docker\cli-plugins\docker-desktop.exe
  dhi: CLI for managing Docker Hardened Images (Docker I

# 3. Validação dos metadados existentes

Aqui vamos responder três perguntas práticas: quais arquivos estão disponíveis, quais colunas existem em cada arquivo e se os valores podem ser lidos corretamente.

O notebook lê os arquivos e mostra os resultados. Nada em `data/raw/` é renomeado, corrigido ou apagado.

In [27]:
from __future__ import annotations

import csv
import hashlib
import io
import math
import re
import unicodedata
from collections import Counter
from pathlib import Path, PurePosixPath
from urllib.parse import urlparse

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Não foi possível localizar a raiz de app/.')


PROJECT_ROOT = find_project_root(Path.cwd())
TABULAR_SUFFIXES = {'.csv', '.tsv'}
NULL_TOKENS = {'null', 'none', 'nan', 'na', 'n/a'}

def normalize_video_name(value: object) -> str:
    text = unicodedata.normalize('NFC', str(value).strip())
    private_use_replacements = {chr(0xF022): '"', chr(0xF03F): '?', chr(0xF05C): '\\'}
    text = ''.join(private_use_replacements.get(char, char) for char in text)
    return text.casefold()
EXPECTED_SCHEMA_BY_FILENAME = {
    'annotations.csv': {
        'video_id': 'string',
        'video_name': 'string',
        'class': 'string',
        'user_id': 'string',
        'width': 'number',
        'height': 'number',
        'fps': 'number',
        'url_page': 'url',
        'url_download': 'url',
    },
    'error.csv': {
        'video_id': 'string',
        'video_name': 'string',
        'class': 'string',
        'user_id': 'string',
        'width': 'number',
        'height': 'number',
        'fps': 'number',
        'url_page': 'url',
        'url_download': 'url',
    },
}
MINDS_EXPECTED_SCHEMA_BY_FILENAME = {
    'annotations.csv': {
        'video_id': 'string',
        'video_name': 'string',
        'class': 'string',
        'user_id': 'string',
        'width': 'number',
        'height': 'number',
        'fps': 'number',
    },
}

# A configuração é uma lista para permitir acrescentar outros datasets sem reescrever o fluxo.
DATASETS = {
    'v-librasil-raw': {
        'root': PROJECT_ROOT / 'data' / 'raw' / 'v-librasil-raw',
        'filesystem_key_column': 'video_name',
        'expected_schema_by_filename': EXPECTED_SCHEMA_BY_FILENAME,
    },
    'minds-libras-raw': {
        'root': PROJECT_ROOT / 'data' / 'raw' / 'minds-libras-raw',
        'filesystem_key_column': 'video_id',
        'expected_schema_by_filename': MINDS_EXPECTED_SCHEMA_BY_FILENAME,
    },
}

{name: str(config['root']) for name, config in DATASETS.items()}

{'v-librasil-raw': 'c:\\Users\\mdbaa\\development\\Dissertacao\\app\\data\\raw\\v-librasil-raw',
 'minds-libras-raw': 'c:\\Users\\mdbaa\\development\\Dissertacao\\app\\data\\raw\\minds-libras-raw'}

## 3.1 O que vamos conferir

1. **Arquivos:** nome, caminho, extensão e tamanho de cada arquivo encontrado.
2. **Colunas:** se o cabeçalho tem `video_id`, `video_name`, `class`, `user_id`, `width`, `height`, `fps`, URLs ou alguma coluna inesperada.
3. **Valores:** se existem campos vazios, nulos, números que não podem ser lidos ou URLs inválidas.
4. **Leitura:** se o arquivo usa uma codificação e um formato de CSV que podem ser interpretados.

Os resultados completos ficam nos DataFrames `file_inventory_df`, `schema_inventory_df`, `column_profile_df` e `validation_issues_df`.

In [28]:
def inventory_files(dataset_name: str, root: Path) -> list[dict]:
    if not root.is_dir():
        return []
    rows = []
    for path in sorted(path for path in root.rglob('*') if path.is_file()):
        relative_path = path.relative_to(root).as_posix()
        suffix = path.suffix.lower() or '<sem_extensão>'
        rows.append({
            'dataset': dataset_name,
            'relative_path': relative_path,
            'suffix': suffix,
            'size_bytes': path.stat().st_size,
            'tabular_candidate': suffix in TABULAR_SUFFIXES,
        })
    return rows


def decode_bytes(raw: bytes) -> dict:
    candidates = ['utf-8-sig', 'utf-8', 'utf-16', 'cp1252', 'latin-1']
    for encoding in candidates:
        try:
            text = raw.decode(encoding)
        except UnicodeDecodeError:
            continue
        replacement_count = text.count(chr(0xFFFD))
        control_count = sum(
            ord(char) < 32 and char not in {'\n', '\r', '\t'}
            for char in text
        )
        is_utf8 = encoding in {'utf-8', 'utf-8-sig'}
        status = 'ok' if is_utf8 and replacement_count == 0 and control_count == 0 else 'attention'
        return {
            'text': text,
            'encoding': encoding,
            'encoding_status': status,
            'replacement_count': replacement_count,
            'control_count': control_count,
        }
    return {
        'text': None,
        'encoding': None,
        'encoding_status': 'undecodable',
        'replacement_count': None,
        'control_count': None,
    }


def detect_dialect(text: str, suffix: str) -> tuple[str, csv.Dialect]:
    default_delimiter = '\t' if suffix == '.tsv' else ','
    try:
        dialect = csv.Sniffer().sniff(text[:65536], delimiters=',;\t|')
        return dialect.delimiter, dialect
    except csv.Error:
        dialect = csv.excel_tab() if default_delimiter == '\t' else csv.excel()
        return default_delimiter, dialect

In [29]:
def is_null(value: str) -> bool:
    return value.strip().casefold() in NULL_TOKENS


def parse_number(value: str) -> float | None:
    try:
        number = float(value.strip())
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None


def is_valid_url(value: str) -> bool:
    if not value or any(char.isspace() for char in value):
        return False
    parsed = urlparse(value)
    return parsed.scheme in {'http', 'https'} and bool(parsed.netloc)


def infer_value_type(value: str) -> str:
    stripped = value.strip()
    if value == '':
        return 'empty'
    if is_null(value):
        return 'null'
    if is_valid_url(value):
        return 'url'
    if parse_number(value) is not None:
        return 'number'
    if stripped.casefold() in {'true', 'false'}:
        return 'boolean'
    return 'string'


def add_issue(issues: list[dict], file_path: str, row_number: int | None, issue: str, column: str | None = None, value: str | None = None, detail: str | None = None) -> None:
    sample = None if value is None else value[:200]
    issues.append({
        'file_path': file_path,
        'row_number': row_number,
        'column': column,
        'issue': issue,
        'value_sample': sample,
        'detail': detail,
    })


def validate_tabular_file(path: Path, root: Path, expected_schema: dict[str, str] | None) -> tuple[dict, list[dict], list[dict]]:
    relative_path = path.relative_to(root).as_posix()
    expected_schema = expected_schema or {}
    raw = path.read_bytes()
    decoded = decode_bytes(raw)
    issues: list[dict] = []
    if decoded['text'] is None:
        add_issue(issues, relative_path, None, 'encoding_undecodable', detail='Nenhuma codificação suportada pôde ser usada.')
        return ({'file_path': relative_path, 'rows': 0, 'column_names': [], 'expected_columns': list(expected_schema), 'missing_columns': list(expected_schema), 'unexpected_columns': [], 'duplicate_columns': [], 'blank_column_names': [], 'encoding': None, 'encoding_status': 'undecodable', 'delimiter': None, 'row_width_errors': 0, 'status': 'attention'}, [], issues)

    text = decoded['text']
    delimiter, dialect = detect_dialect(text, path.suffix.lower())
    try:
        rows = list(csv.reader(io.StringIO(text), dialect))
    except csv.Error as error:
        add_issue(issues, relative_path, None, 'csv_parse_error', detail=str(error))
        rows = []

    if not rows:
        add_issue(issues, relative_path, None, 'empty_tabular_file')
        return ({'file_path': relative_path, 'rows': 0, 'column_names': [], 'expected_columns': list(expected_schema), 'missing_columns': list(expected_schema), 'unexpected_columns': [], 'duplicate_columns': [], 'blank_column_names': [], 'encoding': decoded['encoding'], 'encoding_status': decoded['encoding_status'], 'delimiter': delimiter, 'row_width_errors': 0, 'status': 'attention'}, [], issues)

    header = rows[0]
    data_rows = rows[1:]
    duplicate_columns = sorted(column for column, count in Counter(header).items() if column and count > 1)
    blank_column_names = [f'posição_{index}' for index, column in enumerate(header, start=1) if not column.strip()]
    missing_columns = [column for column in expected_schema if column not in header]
    unexpected_columns = [column for column in header if column not in expected_schema]
    for column in duplicate_columns:
        add_issue(issues, relative_path, 1, 'duplicate_column', column=column)
    for position in [index for index, column in enumerate(header, start=1) if not column.strip()]:
        add_issue(issues, relative_path, 1, 'blank_column_name', column=f'posição_{position}')

    column_stats = [
        {'column': column, 'position': position, 'expected_type': expected_schema.get(column), 'observed_types': Counter(), 'non_empty_count': 0, 'null_count': 0, 'blank_count': 0, 'whitespace_only_count': 0, 'type_error_count': 0, 'invalid_examples': []}
        for position, column in enumerate(header, start=1)
    ]
    row_width_errors = 0
    for row_number, row in enumerate(data_rows, start=2):
        if len(row) != len(header):
            row_width_errors += 1
            add_issue(issues, relative_path, row_number, 'row_width_mismatch', detail=f'esperado={len(header)}, encontrado={len(row)}')
        values = row[:len(header)] + [''] * max(0, len(header) - len(row))
        for index, value in enumerate(values):
            stats = column_stats[index]
            expected_type = stats['expected_type']
            value_type = infer_value_type(value)
            stats['observed_types'][value_type] += 1
            if value == '':
                stats['blank_count'] += 1
                add_issue(issues, relative_path, row_number, 'empty_string', column=stats['column'], value=value)
                continue
            if not value.strip():
                stats['whitespace_only_count'] += 1
                add_issue(issues, relative_path, row_number, 'whitespace_only_string', column=stats['column'], value=value)
                continue
            if is_null(value):
                stats['null_count'] += 1
                add_issue(issues, relative_path, row_number, 'null_value', column=stats['column'], value=value)
                continue
            stats['non_empty_count'] += 1
            type_error = (expected_type == 'number' and parse_number(value) is None) or (expected_type == 'url' and not is_valid_url(value))
            if type_error:
                stats['type_error_count'] += 1
                if len(stats['invalid_examples']) < 5:
                    stats['invalid_examples'].append(value[:120])
                issue_name = 'invalid_number' if expected_type == 'number' else 'invalid_url'
                add_issue(issues, relative_path, row_number, issue_name, column=stats['column'], value=value, detail=f'tipo_esperado={expected_type}')

    column_profiles = []
    for stats in column_stats:
        column_profiles.append({
            'file_path': relative_path,
            'column': stats['column'],
            'position': stats['position'],
            'expected_type': stats['expected_type'],
            'observed_types': dict(stats['observed_types']),
            'non_empty_count': stats['non_empty_count'],
            'null_count': stats['null_count'],
            'blank_count': stats['blank_count'],
            'whitespace_only_count': stats['whitespace_only_count'],
            'type_error_count': stats['type_error_count'],
            'invalid_examples': stats['invalid_examples'],
            'status': 'attention' if stats['null_count'] or stats['blank_count'] or stats['whitespace_only_count'] or stats['type_error_count'] else 'ok',
        })

    if decoded['encoding_status'] == 'attention':
        add_issue(issues, relative_path, None, 'encoding_attention', detail=f"encoding={decoded['encoding']}, replacements={decoded['replacement_count']}, controls={decoded['control_count']}")
    schema_issues = bool(missing_columns or unexpected_columns or duplicate_columns or blank_column_names or row_width_errors)
    report = {
        'file_path': relative_path,
        'rows': len(data_rows),
        'column_count': len(header),
        'column_names': header,
        'expected_columns': list(expected_schema),
        'missing_columns': missing_columns,
        'unexpected_columns': unexpected_columns,
        'duplicate_columns': duplicate_columns,
        'blank_column_names': blank_column_names,
        'encoding': decoded['encoding'],
        'encoding_status': decoded['encoding_status'],
        'delimiter': delimiter,
        'row_width_errors': row_width_errors,
        'schema_status': 'attention' if schema_issues else 'ok',
        'validation_status': 'attention' if issues else 'ok',
        'status': 'attention' if schema_issues or issues else 'ok',
    }
    return report, column_profiles, issues

In [30]:
file_inventory: list[dict] = []
schema_inventory: list[dict] = []
column_profile: list[dict] = []
validation_issues: list[dict] = []
dataset_summary: list[dict] = []

for dataset_name, config in DATASETS.items():
    root = config['root']
    dataset_files = inventory_files(dataset_name, root)
    file_inventory.extend(dataset_files)
    expected_by_filename = config.get('expected_schema_by_filename', {})
    tabular_paths = [
        root / row['relative_path']
        for row in dataset_files
        if row['tabular_candidate']
    ]
    dataset_reports = []
    for path in tabular_paths:
        report, profiles, issues = validate_tabular_file(
            path, root, expected_by_filename.get(path.name)
        )
        schema_inventory.append(report)
        column_profile.extend(profiles)
        validation_issues.extend(issues)
        dataset_reports.append(report)
    dataset_summary.append({
        'dataset': dataset_name,
        'root': str(root),
        'root_exists': root.is_dir(),
        'files_total': len(dataset_files),
        'tabular_files': len(tabular_paths),
        'files_by_suffix': dict(Counter(row['suffix'] for row in dataset_files)),
        'tabular_files_with_attention': sum(report['status'] == 'attention' for report in dataset_reports),
    })

len(file_inventory), len(schema_inventory), len(column_profile), len(validation_issues)

# Tradução apenas para apresentação; os nomes das colunas de origem permanecem intactos.
LABELS_PTBR = {
    'dataset': 'dataset',
    'root': 'raiz',
    'root_exists': 'raiz_existe',
    'relative_path': 'caminho_relativo',
    'file_path': 'arquivo',
    'suffix': 'extensão',
    'size_bytes': 'tamanho_bytes',
    'tabular_candidate': 'candidato_tabelar',
    'files_total': 'quantidade_arquivos',
    'tabular_files': 'quantidade_arquivos_tabelares',
    'files_by_suffix': 'arquivos_por_extensão',
    'tabular_files_with_attention': 'arquivos_tabelares_com_atenção',
    'rows': 'quantidade_linhas',
    'column_count': 'quantidade_colunas',
    'column_names': 'colunas',
    'expected_columns': 'colunas_esperadas',
    'missing_columns': 'colunas_ausentes',
    'unexpected_columns': 'colunas_inesperadas',
    'duplicate_columns': 'colunas_duplicadas',
    'blank_column_names': 'nomes_de_coluna_vazios',
    'encoding': 'codificação',
    'encoding_status': 'status_da_codificação',
    'delimiter': 'delimitador',
    'row_width_errors': 'linhas_com_quantidade_de_campos_incorreta',
    'schema_status': 'status_do_esquema',
    'validation_status': 'status_da_validação',
    'status': 'status_geral',
    'column': 'coluna',
    'position': 'posição',
    'expected_type': 'tipo_esperado',
    'observed_types': 'tipos_observados',
    'non_empty_count': 'quantidade_preenchidos',
    'null_count': 'quantidade_nulos',
    'blank_count': 'quantidade_strings_vazias',
    'whitespace_only_count': 'quantidade_apenas_espaços',
    'type_error_count': 'quantidade_erros_de_tipo',
    'invalid_examples': 'exemplos_inválidos',
    'row_number': 'linha',
    'issue': 'problema',
    'value_sample': 'amostra_do_valor',
    'detail': 'detalhe',
    'total_issues': 'total_de_problemas',
    'issues_by_type': 'problemas_por_tipo',
}
TYPE_LABELS_PTBR = {'string': 'texto', 'number': 'número', 'url': 'URL', 'empty': 'vazio', 'null': 'nulo', 'boolean': 'booleano'}
STATUS_LABELS_PTBR = {'ok': 'ok', 'attention': 'atenção'}
ISSUE_LABELS_PTBR = {
    'encoding_undecodable': 'codificação não interpretável',
    'encoding_attention': 'atenção na codificação',
    'csv_parse_error': 'erro de leitura CSV',
    'empty_tabular_file': 'arquivo tabular vazio',
    'duplicate_column': 'coluna duplicada',
    'blank_column_name': 'nome de coluna vazio',
    'row_width_mismatch': 'quantidade de campos inconsistente',
    'empty_string': 'string vazia',
    'whitespace_only_string': 'string somente com espaços',
    'null_value': 'valor nulo',
    'invalid_number': 'número não interpretável',
    'invalid_url': 'URL inválida',
}

def translate_record(record: dict) -> dict:
    translated = {}
    for key, value in record.items():
        output_key = LABELS_PTBR.get(key, key)
        if key == 'expected_type':
            value = TYPE_LABELS_PTBR.get(value, value)
        elif key == 'observed_types':
            value = {TYPE_LABELS_PTBR.get(name, name): count for name, count in value.items()}
        elif key == 'issue':
            value = ISSUE_LABELS_PTBR.get(value, value)
        elif key == 'issues_by_type':
            value = {ISSUE_LABELS_PTBR.get(name, name): count for name, count in value.items()}
        elif key in {'status', 'schema_status', 'validation_status', 'encoding_status'}:
            value = STATUS_LABELS_PTBR.get(value, value)
        translated[output_key] = value
    return translated

dataset_summary_df = pd.DataFrame([translate_record(record) for record in dataset_summary])
file_inventory_df = pd.DataFrame([translate_record(record) for record in file_inventory])
schema_inventory_df = pd.DataFrame([translate_record(record) for record in schema_inventory])
column_profile_df = pd.DataFrame([translate_record(record) for record in column_profile])
validation_issues_df = pd.DataFrame([translate_record(record) for record in validation_issues])
issues_summary_df = pd.DataFrame([translate_record({
    'total_issues': len(validation_issues),
    'issues_by_type': dict(Counter(issue['issue'] for issue in validation_issues)),
})])

## 3.2 Quais arquivos existem?

A tabela mostra um resumo do que foi encontrado no diretório do dataset. A lista completa está em `file_inventory_df`.

Arquivos `.mp4` são vídeos e não são tratados como tabelas. Arquivos `.csv` e `.tsv` são os que entram na conferência de colunas. Se aparecerem arquivos `.metadata` ou `.incomplete`, eles pertencem ao cache do download e ficam registrados, mas não são analisados como dados.

In [31]:
dataset_summary_df

,dataset,raiz,raiz_existe,quantidade_arquivos,quantidade_arquivos_tabelares,arquivos_por_extensão,arquivos_tabelares_com_atenção
0,v-librasil-raw,c:\Users\mdbaa\development\Dissertacao\app\dat...,True,8189,2,"{'<sem_extensão>': 2, '.tag': 1, '.metadata': ...",0
1,minds-libras-raw,c:\Users\mdbaa\development\Dissertacao\app\dat...,True,1622,1,"{'<sem_extensão>': 2, '.tag': 1, '.metadata': ...",0


## 3.3 Quais colunas cada arquivo tem?

Cada linha representa um arquivo tabular. A coluna `colunas` mostra o cabeçalho exatamente como foi encontrado no arquivo.

- `colunas_ausentes` indica o que era esperado, mas não apareceu.
- `colunas_inesperadas` indica campos adicionais que não estavam previstos.
- `colunas_duplicadas` indica nomes repetidos no cabeçalho.
- `status_do_esquema = ok` significa que o cabeçalho e a estrutura das linhas estão consistentes.

In [32]:
schema_inventory_df

,arquivo,quantidade_linhas,quantidade_colunas,colunas,colunas_esperadas,colunas_ausentes,colunas_inesperadas,colunas_duplicadas,nomes_de_coluna_vazios,codificação,status_da_codificação,delimitador,linhas_com_quantidade_de_campos_incorreta,status_do_esquema,status_da_validação,status_geral
0,annotations.csv,4086,9,"[video_id, video_name, class, user_id, width, ...","[video_id, video_name, class, user_id, width, ...",[],[],[],[],utf-8-sig,ok,",",0,ok,ok,ok
1,error.csv,3,9,"[video_id, video_name, class, user_id, width, ...","[video_id, video_name, class, user_id, width, ...",[],[],[],[],utf-8-sig,ok,",",0,ok,ok,ok
2,annotations.csv,800,7,"[video_id, video_name, class, user_id, width, ...","[video_id, video_name, class, user_id, width, ...",[],[],[],[],utf-8-sig,ok,",",0,ok,ok,ok


## 3.4 Os valores estão preenchidos e no formato esperado?

Cada linha do perfil representa uma coluna de um arquivo. Use estas colunas para entender o resultado:

- `tipo_esperado` mostra o formato que estamos esperando.
- `tipos_observados` mostra o que foi encontrado nos valores.
- `quantidade_nulos` e `quantidade_strings_vazias` mostram campos sem conteúdo.
- `quantidade_erros_de_tipo` mostra números ou URLs que não puderam ser interpretados.
- `status_geral = ok` indica que não foi encontrado problema naquela coluna.

Quando houver problemas, eles aparecem com mais detalhes em `validation_issues_df`, incluindo o arquivo, a linha e uma amostra do valor.

In [33]:
column_profile_df

,arquivo,coluna,posição,tipo_esperado,tipos_observados,quantidade_preenchidos,quantidade_nulos,quantidade_strings_vazias,quantidade_apenas_espaços,quantidade_erros_de_tipo,exemplos_inválidos,status_geral
0,annotations.csv,video_id,1,texto,{'texto': 4086},4086,0,0,0,0,[],ok
1,annotations.csv,video_name,2,texto,{'texto': 4086},4086,0,0,0,0,[],ok
2,annotations.csv,class,3,texto,{'texto': 4086},4086,0,0,0,0,[],ok
3,annotations.csv,user_id,4,texto,{'texto': 4086},4086,0,0,0,0,[],ok
4,annotations.csv,width,5,número,{'número': 4086},4086,0,0,0,0,[],ok
5,annotations.csv,height,6,número,{'número': 4086},4086,0,0,0,0,[],ok
6,annotations.csv,fps,7,número,{'número': 4086},4086,0,0,0,0,[],ok
7,annotations.csv,url_page,8,URL,{'URL': 4086},4086,0,0,0,0,[],ok
8,annotations.csv,url_download,9,URL,{'URL': 4086},4086,0,0,0,0,[],ok
9,error.csv,video_id,1,texto,{'texto': 3},3,0,0,0,0,[],ok


# 4. Validação das chaves e da unicidade

Agora verificamos se os registros podem ser identificados sem ambiguidade. Esta etapa procura valores repetidos e conflitos entre as colunas, mas não decide se a quantidade de exemplos por classe é boa ou ruim.

Serão conferidos: `video_id`, `video_name`, `url_download`, a associação entre `video_id` e `class`, e repetições da combinação `class + user_id`.

In [34]:
def load_integrity_records(dataset_name: str, config: dict) -> list[dict]:
    root = config['root']
    records = []
    for item in file_inventory:
        if item['dataset'] != dataset_name or not item['tabular_candidate']:
            continue
        path = root / item['relative_path']
        decoded = decode_bytes(path.read_bytes())
        if decoded['text'] is None:
            continue
        _, dialect = detect_dialect(decoded['text'], path.suffix.lower())
        reader = csv.DictReader(io.StringIO(decoded['text']), dialect=dialect)
        for line_number, record in enumerate(reader, start=2):
            clean_record = {key: value for key, value in record.items() if key is not None}
            clean_record['_arquivo'] = item['relative_path']
            clean_record['_dataset'] = dataset_name
            clean_record['_linha'] = line_number
            records.append(clean_record)
    return records


integrity_records = []
for dataset_name, config in DATASETS.items():
    integrity_records.extend(load_integrity_records(dataset_name, config))
integrity_df = pd.DataFrame(integrity_records)

integrity_summary_rows = []
integrity_detail_rows = []


def valid_key_rows(columns: list[str]) -> pd.DataFrame:
    if integrity_df.empty or any(column not in integrity_df.columns for column in columns):
        return pd.DataFrame()
    valid = integrity_df.copy()
    mask = pd.Series(True, index=valid.index)
    for column in columns:
        mask &= valid[column].notna() & valid[column].astype(str).str.strip().ne('')
    return valid.loc[mask]


def check_repeated_key(label: str, columns: list[str], note: str) -> None:
    valid = valid_key_rows(columns)
    if valid.empty:
        integrity_summary_rows.append({
            'verificação': label,
            'colunas_usadas': ' + '.join(columns),
            'quantidade_grupos_repetidos': 0,
            'quantidade_linhas_envolvidas': 0,
            'status': 'não verificado',
            'observação': 'Coluna ausente ou sem valores preenchidos.',
        })
        return
    grouped = valid.groupby(columns, dropna=False, sort=True).agg(
        quantidade_linhas=('_linha', 'size'),
        arquivos=('_arquivo', lambda values: ', '.join(sorted(set(values)))),
    ).reset_index()
    repeated = grouped[grouped['quantidade_linhas'] > 1].copy()
    integrity_summary_rows.append({
        'verificação': label,
        'colunas_usadas': ' + '.join(columns),
        'quantidade_grupos_repetidos': len(repeated),
        'quantidade_linhas_envolvidas': int(repeated['quantidade_linhas'].sum()),
        'status': 'ok' if repeated.empty else 'atenção',
        'observação': note,
    })
    for _, row in repeated.iterrows():
        key = ' | '.join(f'{column}={row[column]}' for column in columns)
        integrity_detail_rows.append({
            'verificação': label,
            'chave': key,
            'quantidade_linhas': int(row['quantidade_linhas']),
            'arquivos': row['arquivos'],
            'observação': note,
        })


check_repeated_key('video_id repetido', ['video_id'], 'Duas ou mais linhas apontam para o mesmo identificador.')
check_repeated_key('video_name repetido', ['video_name'], 'Duas ou mais linhas usam o mesmo nome de vídeo.')
check_repeated_key('mesmo arquivo em url_download', ['url_download'], 'Duas ou mais linhas apontam para a mesma URL de download.')
check_repeated_key('combinação class + user_id repetida', ['class', 'user_id'], 'A repetição é registrada para investigação; não é classificada como boa ou ruim nesta etapa.')

valid = valid_key_rows(['video_id', 'class'])
if valid.empty:
    integrity_summary_rows.append({
        'verificação': 'video_id associado a classes diferentes',
        'colunas_usadas': 'video_id + class',
        'quantidade_grupos_repetidos': 0,
        'quantidade_linhas_envolvidas': 0,
        'status': 'não verificado',
        'observação': 'Coluna ausente ou sem valores preenchidos.',
    })
else:
    classes_por_video = valid.groupby('video_id', sort=True).agg(
        quantidade_classes=('class', 'nunique'),
        classes=('class', lambda values: ', '.join(sorted(set(values)))),
        quantidade_linhas=('class', 'size'),
    ).reset_index()
    conflicts = classes_por_video[classes_por_video['quantidade_classes'] > 1]
    integrity_summary_rows.append({
        'verificação': 'video_id associado a classes diferentes',
        'colunas_usadas': 'video_id + class',
        'quantidade_grupos_repetidos': len(conflicts),
        'quantidade_linhas_envolvidas': int(conflicts['quantidade_linhas'].sum()),
        'status': 'ok' if conflicts.empty else 'atenção',
        'observação': 'Cada video_id deve apontar para uma única classe.',
    })
    for _, row in conflicts.iterrows():
        integrity_detail_rows.append({
            'verificação': 'video_id associado a classes diferentes',
            'chave': f"video_id={row['video_id']}",
            'quantidade_linhas': int(row['quantidade_linhas']),
            'arquivos': '',
            'observação': f"classes encontradas: {row['classes']}",
        })

chaves_unicidade_df = pd.DataFrame(integrity_summary_rows)
conflitos_integridade_df = pd.DataFrame(integrity_detail_rows)

## 4.1 Resultado das verificações

`chaves_unicidade_df` resume cada teste. O status `ok` significa que não houve repetição ou conflito naquele teste. O status `atenção` indica que existem linhas para investigar.

`conflitos_integridade_df` mostra quais chaves foram repetidas ou associadas a mais de uma classe.

In [35]:
chaves_unicidade_df

,verificação,colunas_usadas,quantidade_grupos_repetidos,quantidade_linhas_envolvidas,status,observação
0,video_id repetido,video_id,0,0,ok,Duas ou mais linhas apontam para o mesmo ident...
1,video_name repetido,video_name,0,0,ok,Duas ou mais linhas usam o mesmo nome de vídeo.
2,mesmo arquivo em url_download,url_download,0,0,ok,Duas ou mais linhas apontam para a mesma URL d...
3,combinação class + user_id repetida,class + user_id,160,800,atenção,A repetição é registrada para investigação; nã...
4,video_id associado a classes diferentes,video_id + class,0,0,ok,Cada video_id deve apontar para uma única classe.


In [36]:
conflitos_integridade_df

,verificação,chave,quantidade_linhas,arquivos,observação
0,combinação class + user_id repetida,class=Acontecer | user_id=Sinalizador01,5,annotations.csv,A repetição é registrada para investigação; nã...
1,combinação class + user_id repetida,class=Acontecer | user_id=Sinalizador02,5,annotations.csv,A repetição é registrada para investigação; nã...
2,combinação class + user_id repetida,class=Acontecer | user_id=Sinalizador05,5,annotations.csv,A repetição é registrada para investigação; nã...
3,combinação class + user_id repetida,class=Acontecer | user_id=Sinalizador06,5,annotations.csv,A repetição é registrada para investigação; nã...
4,combinação class + user_id repetida,class=Acontecer | user_id=Sinalizador08,5,annotations.csv,A repetição é registrada para investigação; nã...
...,...,...,...,...,...
155,combinação class + user_id repetida,class=Vontade | user_id=Sinalizador06,5,annotations.csv,A repetição é registrada para investigação; nã...
156,combinação class + user_id repetida,class=Vontade | user_id=Sinalizador08,5,annotations.csv,A repetição é registrada para investigação; nã...
157,combinação class + user_id repetida,class=Vontade | user_id=Sinalizador10,5,annotations.csv,A repetição é registrada para investigação; nã...
158,combinação class + user_id repetida,class=Vontade | user_id=Sinalizador11,5,annotations.csv,A repetição é registrada para investigação; nã...


# 5. Validação CSV × filesystem

Aqui comparamos o que está registrado nos CSVs com o que realmente existe na pasta `videos/`. A chave física é específica por dataset: `video_name` no V-LIBRASIL e `video_id` no MINDS-Libras.

Os arquivos listados em `error.csv` são tratados como exceções explícitas: eles não entram como ausentes ou não registrados. Um arquivo presente, mas com estrutura MP4 inválida ou tamanho zero, aparece separadamente como inválido ou corrompido.

In [37]:
def inspect_video_file(path: Path) -> tuple[bool, str]:
    try:
        size = path.stat().st_size
        if size == 0:
            return False, 'arquivo vazio'
        with path.open('rb') as stream:
            head = stream.read(1024 * 1024)
            stream.seek(max(size - 1024 * 1024, 0))
            tail = stream.read(1024 * 1024)
        sample = head + tail
        if b'ftyp' not in head[:256]:
            return False, 'assinatura MP4 ftyp ausente'
        if b'moov' not in sample:
            return False, 'estrutura MP4 moov ausente'
        if b'mdat' not in sample:
            return False, 'estrutura MP4 mdat ausente'
        return True, 'estrutura MP4 mínima encontrada'
    except OSError as error:
        return False, f'erro ao ler arquivo: {error}'


video_health_rows = []
for dataset_name, config in DATASETS.items():
    video_dir = config['root'] / 'videos'
    video_paths = sorted(path for path in video_dir.rglob('*') if path.is_file()) if video_dir.is_dir() else []
    for path in video_paths:
        valid, reason = inspect_video_file(path)
        video_health_rows.append({
            'dataset': dataset_name,
            'video_name': path.name,
            'caminho': path.relative_to(config['root']).as_posix(),
            'tamanho_bytes': path.stat().st_size,
            'arquivo_valido': valid,
            'motivo': reason,
        })
video_health_df = pd.DataFrame(video_health_rows)

filesystem_summary_rows = []
filesystem_detail_rows = []
filesystem_exception_rows = []
filesystem_alias_rows = []

for dataset_name, config in DATASETS.items():
    dataset_records = integrity_df[integrity_df['_dataset'] == dataset_name] if not integrity_df.empty else pd.DataFrame()
    annotations_records = dataset_records[dataset_records['_arquivo'] == 'annotations.csv'] if not dataset_records.empty else pd.DataFrame()
    error_records = dataset_records[dataset_records['_arquivo'] == 'error.csv'] if not dataset_records.empty else pd.DataFrame()
    filesystem_key_column = config['filesystem_key_column']
    annotations_names = set(annotations_records[filesystem_key_column].dropna().astype(str).str.strip()) if filesystem_key_column in annotations_records else set()
    error_names = set(error_records[filesystem_key_column].dropna().astype(str).str.strip()) if filesystem_key_column in error_records else set()
    annotations_by_normalized = {}
    for name in annotations_names:
        annotations_by_normalized.setdefault(normalize_video_name(name), set()).add(name)
    error_normalized = {normalize_video_name(name) for name in error_names}
    physical_records = video_health_df[video_health_df['dataset'] == dataset_name]
    physical_names = set(physical_records['video_name'])
    valid_names = set(physical_records.loc[physical_records['arquivo_valido'], 'video_name'])
    invalid_names = set(physical_records.loc[~physical_records['arquivo_valido'], 'video_name'])
    physical_by_normalized = {}
    for name in physical_names:
        physical_by_normalized.setdefault(normalize_video_name(name), set()).add(name)
    valid_normalized = {normalize_video_name(name) for name in valid_names}
    expected_names = annotations_names - error_names
    expected_normalized = {normalize_video_name(name) for name in expected_names}
    registered_existing = {physical_name for key in expected_normalized & valid_normalized for physical_name in physical_by_normalized[key]}
    registered_missing = {name for name in expected_names if normalize_video_name(name) not in physical_by_normalized}
    existing_unregistered = {name for name in valid_names if normalize_video_name(name) not in expected_normalized and normalize_video_name(name) not in error_normalized}
    for normalized_name in (sorted(expected_normalized & set(physical_by_normalized)) if dataset_name == 'v-librasil-raw' else []):
        for annotation_name in sorted(annotations_by_normalized.get(normalized_name, set())):
            for physical_name in sorted(physical_by_normalized[normalized_name]):
                if annotation_name != physical_name:
                    filesystem_alias_rows.append({
                        'dataset': dataset_name,
                        'annotations_video_name': annotation_name,
                        'filesystem_video_name': physical_name,
                        'normalizada_igual': True,
                        'classificacao': 'mesmo vídeo; diferença de encoding/escape/nome',
                    })

    categories = [
        ('registrados e existentes', registered_existing, 'video_name aparece no annotations.csv e existe como arquivo válido.'),
        ('registrados mas ausentes', registered_missing, 'video_name aparece no annotations.csv, mas não existe em videos/.'),
        ('existentes mas não registrados', existing_unregistered, 'arquivo válido existe em videos/, mas não aparece no annotations.csv nem no error.csv.'),
        ('inválidos ou corrompidos', invalid_names, 'arquivo existe, mas falhou na verificação estrutural mínima do MP4.'),
    ]
    for category, names, explanation in categories:
        filesystem_summary_rows.append({
            'dataset': dataset_name,
            'categoria': category,
            'quantidade': len(names),
            'observação': explanation,
        })
        for name in sorted(names):
            filesystem_detail_rows.append({
                'dataset': dataset_name,
                'categoria': category,
                'video_name': name,
            })
    for name in sorted(error_names):
        filesystem_exception_rows.append({
            'dataset': dataset_name,
            'video_name': name,
            'origem': 'error.csv',
            'situação_física': 'presente em videos/' if name in physical_names else 'ausente em videos/',
            'observação': 'Exceção explícita: não entra nos conjuntos de ausência ou não registro.',
        })

filesystem_reconciliation_df = pd.DataFrame(filesystem_summary_rows)
filesystem_reconciliation_details_df = pd.DataFrame(filesystem_detail_rows)
filesystem_error_exceptions_df = pd.DataFrame(filesystem_exception_rows)
filesystem_aliases_df = pd.DataFrame(filesystem_alias_rows)

## 5.1 Resultado da comparação

A tabela abaixo contém os quatro números principais. Para uma coincidência perfeita, a maior parte deve estar em `registrados e existentes`, enquanto as outras categorias devem ficar zeradas, salvo exceções documentadas em `error.csv`.

Os nomes individuais encontrados em cada categoria ficam em `filesystem_reconciliation_details_df`. Os nomes listados no `error.csv` ficam separados em `filesystem_error_exceptions_df`.

In [38]:
filesystem_reconciliation_df

,dataset,categoria,quantidade,observação
0,v-librasil-raw,registrados e existentes,4083,video_name aparece no annotations.csv e existe...
1,v-librasil-raw,registrados mas ausentes,3,"video_name aparece no annotations.csv, mas não..."
2,v-librasil-raw,existentes mas não registrados,3,"arquivo válido existe em videos/, mas não apar..."
3,v-librasil-raw,inválidos ou corrompidos,0,"arquivo existe, mas falhou na verificação estr..."
4,minds-libras-raw,registrados e existentes,800,video_name aparece no annotations.csv e existe...
5,minds-libras-raw,registrados mas ausentes,0,"video_name aparece no annotations.csv, mas não..."
6,minds-libras-raw,existentes mas não registrados,0,"arquivo válido existe em videos/, mas não apar..."
7,minds-libras-raw,inválidos ou corrompidos,0,"arquivo existe, mas falhou na verificação estr..."


In [39]:
filesystem_reconciliation_details_df

,dataset,categoria,video_name
0,v-librasil-raw,registrados e existentes,Abacaxi_Articulador1.mp4
1,v-librasil-raw,registrados e existentes,Abacaxi_Articulador2.mp4
2,v-librasil-raw,registrados e existentes,Abacaxi_Articulador3.mp4
3,v-librasil-raw,registrados e existentes,Abanar_Articulador1.mp4
4,v-librasil-raw,registrados e existentes,Abanar_Articulador2.mp4
...,...,...,...
4884,minds-libras-raw,registrados e existentes,20VontadeSinalizador12-1.mp4
4885,minds-libras-raw,registrados e existentes,20VontadeSinalizador12-2.mp4
4886,minds-libras-raw,registrados e existentes,20VontadeSinalizador12-3.mp4
4887,minds-libras-raw,registrados e existentes,20VontadeSinalizador12-4.mp4


In [40]:
filesystem_error_exceptions_df

,dataset,video_name,origem,situação_física,observação
0,v-librasil-raw,Congelar_Articulador3.mp4,error.csv,ausente em videos/,Exceção explícita: não entra nos conjuntos de ...
1,v-librasil-raw,Criança_Articulador3.mp4,error.csv,ausente em videos/,Exceção explícita: não entra nos conjuntos de ...
2,v-librasil-raw,Poesia_Articulador2.mp4,error.csv,ausente em videos/,Exceção explícita: não entra nos conjuntos de ...


# 6. Validação dos metadados técnicos CSV × vídeo real

Esta etapa chama a validação reutilizável dentro do Docker de processamento. Ela compara `width`, `height` e `fps` do CSV com os valores observados pelo `ffprobe`, sem alterar os arquivos em `data/raw/`.

A saída JSON é a fonte de verdade; os CSVs derivados são carregados para inspeção no notebook.

In [41]:
import json
import subprocess

METADATA_DATASET = DATASET_TO_RUN
METADATA_OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'metadata-validation' / METADATA_DATASET
METADATA_COMPOSE_FILE = PROJECT_ROOT / 'docker' / 'video-processing' / 'compose.yaml'
VALIDATION_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'metadata-validation'
VALIDATION_REGISTRY_PATH = VALIDATION_ROOT / 'validation_runs.json'
VALIDATION_LOGIC_VERSION = 'notebook-validation.v2'
CONTAINER_PROJECT_ROOT = PurePosixPath('/workspace/app')

def docker_path(path: Path) -> str:
    relative = path.resolve().relative_to(PROJECT_ROOT.resolve())
    return str(CONTAINER_PROJECT_ROOT / relative.as_posix())

def load_validation_registry() -> dict:
    if not VALIDATION_REGISTRY_PATH.is_file():
        return {'schema_version': 'validation-run-registry.v1', 'runs': {}}
    return json.loads(VALIDATION_REGISTRY_PATH.read_text(encoding='utf-8'))


def save_validation_registry(registry: dict) -> None:
    VALIDATION_REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
    VALIDATION_REGISTRY_PATH.write_text(
        json.dumps(registry, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )


def dataset_input_fingerprint(dataset_name: str) -> str:
    root = DATASETS[dataset_name]['root']
    signature = []
    for path in sorted(path for path in root.rglob('*') if path.is_file()):
        stat = path.stat()
        signature.append(f'{path.relative_to(root).as_posix()}|{stat.st_size}|{stat.st_mtime_ns}')
    return hashlib.sha256('\n'.join(signature).encode('utf-8')).hexdigest()


def metadata_artifacts_exist(dataset_name: str) -> bool:
    root = VALIDATION_ROOT / dataset_name
    return all((root / name).is_file() for name in (
        'metadata_validation.json', 'video_metadata.csv',
        'metadata_comparison.csv', 'fps_hypotheses_summary.csv',
    ))


def quality_artifacts_exist(dataset_name: str) -> bool:
    manifest_name = 'v_librasil_validated.parquet' if dataset_name == 'v-librasil-raw' else 'minds_libras_validated.parquet'
    root = VALIDATION_ROOT / dataset_name
    return all((root / path).is_file() for path in (
        'reports/metadata_validation_report.md',
        'reports/metadata_issues.csv',
        'reports/metadata_summary.json',
        f'manifests/{manifest_name}',
    ))


def can_reuse_validation(registry: dict, dataset_name: str, stage: str, fingerprint: str) -> bool:
    if FORCE_RERUN:
        return False
    entry = registry.get('runs', {}).get(dataset_name, {})
    return (
        entry.get(f'{stage}_status') == 'completed'
        and entry.get('logic_version') == VALIDATION_LOGIC_VERSION
        and entry.get('input_fingerprint') == fingerprint
        and (metadata_artifacts_exist(dataset_name) if stage == 'metadata' else quality_artifacts_exist(dataset_name))
    )


def update_validation_registry(registry: dict, dataset_name: str, fingerprint: str, **values) -> None:
    entry = registry.setdefault('runs', {}).setdefault(dataset_name, {})
    entry.update({
        'dataset': dataset_name,
        'logic_version': VALIDATION_LOGIC_VERSION,
        'input_fingerprint': fingerprint,
        **values,
    })
    save_validation_registry(registry)


def run_metadata_validation_in_docker() -> None:
    command = [
        'docker', 'compose', '-f', str(METADATA_COMPOSE_FILE),
        'run', '--rm', '-T', 'video-processing',
        'python', '-m', 'libras_translation.cli',
        'video', 'validate-metadata', '--dataset', METADATA_DATASET,
    ]
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, shell=False, capture_output=True,
        text=True, check=False, encoding='utf-8', errors='replace',
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f'Validação de metadados falhou com código {completed.returncode}.')

validation_registry = load_validation_registry()
validation_fingerprint = dataset_input_fingerprint(METADATA_DATASET)
if RUN_METADATA_VALIDATION:
    if can_reuse_validation(validation_registry, METADATA_DATASET, 'metadata', validation_fingerprint):
        print(f'Validação técnica reutilizada: {METADATA_DATASET}')
    else:
        run_metadata_validation_in_docker()
        update_validation_registry(
            validation_registry, METADATA_DATASET, validation_fingerprint,
            metadata_status='completed',
            metadata_artifacts=str(METADATA_OUTPUT_ROOT),
        )

metadata_validation_json_path = METADATA_OUTPUT_ROOT / 'metadata_validation.json'
metadata_validation = json.loads(metadata_validation_json_path.read_text(encoding='utf-8'))
metadata_validation_summary_df = pd.DataFrame([metadata_validation['summary']])
metadata_video_df = pd.read_csv(METADATA_OUTPUT_ROOT / 'video_metadata.csv')
metadata_comparison_df = pd.read_csv(METADATA_OUTPUT_ROOT / 'metadata_comparison.csv')
fps_hypotheses_summary_df = pd.read_csv(METADATA_OUTPUT_ROOT / 'fps_hypotheses_summary.csv')
print(f'Dataset selecionado: {METADATA_DATASET}')
print(f'Registro de execuções: {VALIDATION_REGISTRY_PATH}')

Validação técnica reutilizada: minds-libras-raw
Dataset selecionado: minds-libras-raw
Registro de execuções: c:\Users\mdbaa\development\Dissertacao\app\data\processed\metadata-validation\validation_runs.json


## 6.1 Resultado da validação técnica

`fps_rate_status` testa `csv.fps == actual_fps`; `fps_frame_count_status` testa `csv.fps == actual_frame_count`. A coluna original `fps` permanece intacta. `inconclusive` indica que um valor necessário não estava disponível; `none` indica que os dois valores confiáveis não coincidiram.

In [42]:
display(metadata_validation_summary_df)
display(fps_hypotheses_summary_df)
display(metadata_comparison_df.head())

,dataset,videos_seen,videos_ok,videos_partial,videos_with_probe_error,width_matches,height_matches,fps_rate_matches,fps_frame_count_matches,fps_none_matches,fps_inconclusive,fps_hypothesis_counts,dataset_fps_interpretation,fps_tolerance
0,minds-libras-raw,800,800,0,0,800,800,0,800,0,0,{'frame_count': 800},frame_count,0.01


,hypothesis,count
0,frame_count,800


,relative_path,csv_width,csv_height,csv_fps_raw,csv_fps_numeric,width_status,height_status,fps_rate_status,fps_frame_count_status,fps_hypothesis,overall_status
0,01AcontecerSinalizador01-1.mp4,1920,1080,104.0,104.0,match,match,mismatch,match,frame_count,match
1,01AcontecerSinalizador01-2.mp4,1920,1080,130.0,130.0,match,match,mismatch,match,frame_count,match
2,01AcontecerSinalizador01-3.mp4,1920,1080,120.0,120.0,match,match,mismatch,match,frame_count,match
3,01AcontecerSinalizador01-4.mp4,1920,1080,121.0,121.0,match,match,mismatch,match,frame_count,match
4,01AcontecerSinalizador01-5.mp4,1920,1080,131.0,131.0,match,match,mismatch,match,frame_count,match


# 7. Validação da convenção do nome do arquivo

O nome do arquivo deve carregar os mesmos metadados registrados no CSV. Para um arquivo como `Abacaxi_Articulador1.mp4`, o nome indica `class = Abacaxi` e `user_id = Articulador1`.

A comparação abaixo procura diferenças entre o que o nome informa e o que está em `annotations.csv` ou `error.csv`. Também verifica arquivos físicos que ainda não possuem uma linha no CSV.

In [43]:
def parse_video_filename(video_name: str, dataset_name: str) -> tuple[str | None, str | None, str | None]:
    if dataset_name != 'v-librasil-raw':
        return None, None, 'convenção de nome não especificada para este dataset'
    path = Path(str(video_name))
    if path.suffix.casefold() != '.mp4':
        return None, None, 'extensão diferente de .mp4'
    if '_' not in path.stem:
        return None, None, 'nome não contém separador class_user_id'
    filename_class, filename_user = path.stem.rsplit('_', 1)
    if not filename_class or not re.fullmatch(r'Articulador\d+', filename_user):
        return filename_class or None, filename_user or None, 'user_id no nome não segue Articulador + número'
    return filename_class, filename_user, None


filename_convention_rows = []

for dataset_name in DATASETS:
    physical = video_health_df[video_health_df['dataset'] == dataset_name]
    for _, row in physical.iterrows():
        filename_class, filename_user, problem = parse_video_filename(row['video_name'], dataset_name)
        filename_convention_rows.append({
            'dataset': dataset_name,
            'origem': 'filesystem',
            'linha': None,
            'video_name': row['video_name'],
            'class_do_nome': filename_class,
            'user_id_do_nome': filename_user,
            'class_registrada': None,
            'user_id_registrado': None,
            'status': 'não aplicável' if problem and dataset_name != 'v-librasil-raw' else ('atenção' if problem else 'ok'),
            'problema': problem or 'nome estruturalmente válido; sem registro CSV para comparar',
        })

    registered = integrity_df[(integrity_df['_dataset'] == dataset_name) & integrity_df['_arquivo'].isin(['annotations.csv', 'error.csv'])] if not integrity_df.empty else pd.DataFrame()
    for _, row in registered.iterrows():
        filename_class, filename_user, problem = parse_video_filename(row.get('video_name', ''), dataset_name)
        problems = []
        if problem:
            problems.append(problem)
        if not problem and filename_class != row.get('class'):
            problems.append(f"class registrada={row.get('class')!r}, class do nome={filename_class!r}")
        if not problem and filename_user != row.get('user_id'):
            problems.append(f"user_id registrado={row.get('user_id')!r}, user_id do nome={filename_user!r}")
        filename_convention_rows.append({
            'dataset': dataset_name,
            'origem': row['_arquivo'],
            'linha': row['_linha'],
            'video_name': row.get('video_name'),
            'class_do_nome': filename_class,
            'user_id_do_nome': filename_user,
            'class_registrada': row.get('class'),
            'user_id_registrado': row.get('user_id'),
            'status': 'não aplicável' if problem and dataset_name != 'v-librasil-raw' else ('atenção' if problems else 'ok'),
            'problema': '; '.join(problems),
        })

filename_convention_df = pd.DataFrame(filename_convention_rows)
filename_convention_summary_df = (
    filename_convention_df.groupby(['origem', 'status'], dropna=False)
    .size()
    .reset_index(name='quantidade')
)

## 7.1 Resultado da convenção dos nomes

`status = ok` significa que o nome segue a convenção e, quando há registro CSV, bate com `class` e `user_id`. As divergências aparecem em `filename_convention_df`, com o número da linha do CSV e os dois valores para comparação.

In [44]:
filename_convention_summary_df

,origem,status,quantidade
0,annotations.csv,atenção,6
1,annotations.csv,não aplicável,800
2,annotations.csv,ok,4080
3,error.csv,ok,3
4,filesystem,não aplicável,800
5,filesystem,ok,4086


In [45]:
filename_convention_df[filename_convention_df['status'] == 'atenção']

,dataset,origem,linha,video_name,class_do_nome,user_id_do_nome,class_registrada,user_id_registrado,status,problema
4817,v-librasil-raw,annotations.csv,733.0,"Cérebro (de ""ervilha"""")_Articulador3.mp4""",NaN,NaN,"Cérebro (de ""ervilha"""")""",Articulador3,atenção,extensão diferente de .mp4
4818,v-librasil-raw,annotations.csv,734.0,"Cérebro (de ""ervilha"""")_Articulador2.mp4""",NaN,NaN,"Cérebro (de ""ervilha"""")""",Articulador2,atenção,extensão diferente de .mp4
4819,v-librasil-raw,annotations.csv,735.0,"Cérebro (de ""ervilha"""")_Articulador1.mp4""",NaN,NaN,"Cérebro (de ""ervilha"""")""",Articulador1,atenção,extensão diferente de .mp4
5895,v-librasil-raw,annotations.csv,1811.0,Frente\Á Frente_Articulador3.mp4,Á Frente,Articulador3,Frente\Á Frente,Articulador3,atenção,"class registrada='Frente\\Á Frente', class do ..."
5896,v-librasil-raw,annotations.csv,1812.0,Frente\Á Frente_Articulador2.mp4,Á Frente,Articulador2,Frente\Á Frente,Articulador2,atenção,"class registrada='Frente\\Á Frente', class do ..."
5897,v-librasil-raw,annotations.csv,1813.0,Frente\Á Frente_Articulador1.mp4,Á Frente,Articulador1,Frente\Á Frente,Articulador1,atenção,"class registrada='Frente\\Á Frente', class do ..."


# 8. Validação de proveniência, decisão e manifest derivado

Esta etapa vem depois da convenção de nomes. Ela reconcilia `error.csv` com `annotations.csv` e `videos/`, procura ausências silenciosas e produz evidências auditáveis. `GO` significa que não foram encontrados problemas `CRITICAL` ou `ERROR`; `NO-GO` não apaga nem altera o dataset original.

In [46]:
QUALITY_ROOT = VALIDATION_ROOT / METADATA_DATASET
QUALITY_REPORT_ROOT = QUALITY_ROOT / 'reports'
QUALITY_MANIFEST_ROOT = QUALITY_ROOT / 'manifests'
QUALITY_MANIFEST_NAME = 'v_librasil_validated.parquet' if METADATA_DATASET == 'v-librasil-raw' else 'minds_libras_validated.parquet'

def run_quality_validation_in_docker() -> None:
    command = [
        'docker', 'compose', '-f', str(METADATA_COMPOSE_FILE),
        'run', '--rm', '-T', 'video-processing',
        'python', '-m', 'libras_translation.cli',
        'video', 'validate-quality', '--dataset', METADATA_DATASET,
        '--metadata-output', docker_path(METADATA_OUTPUT_ROOT),
        '--output-root', docker_path(QUALITY_ROOT),
    ]
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, shell=False, check=False,
        encoding='utf-8', errors='replace',
    )
    if completed.returncode not in (0, 2):
        raise RuntimeError(f'Geração dos artefatos de qualidade falhou com código {completed.returncode}.')

if not RUN_QUALITY_VALIDATION and not quality_artifacts_exist(METADATA_DATASET):
    raise RuntimeError('A validação de qualidade está desabilitada, mas seus artefatos ainda não existem.')
if not RUN_QUALITY_VALIDATION:
    print(f'Validação de qualidade desabilitada; artefatos existentes serão reutilizados: {METADATA_DATASET}')
elif can_reuse_validation(validation_registry, METADATA_DATASET, 'quality', validation_fingerprint):
    print(f'Validação de qualidade reutilizada: {METADATA_DATASET}')
else:
    run_quality_validation_in_docker()
    quality_summary_path = QUALITY_REPORT_ROOT / 'metadata_summary.json'
    quality_summary = json.loads(quality_summary_path.read_text(encoding='utf-8'))
    update_validation_registry(
        validation_registry, METADATA_DATASET, validation_fingerprint,
        status='completed' if quality_summary.get('decision') == 'GO' else 'completed_with_issues',
        quality_status='completed',
        decision=quality_summary.get('decision'),
        reason=f"Validação técnica e de qualidade executadas; decisão={quality_summary.get('decision')}.",
        quality_artifacts=str(QUALITY_ROOT),
    )
metadata_summary = json.loads((QUALITY_REPORT_ROOT / 'metadata_summary.json').read_text(encoding='utf-8'))
metadata_issues_df = pd.read_csv(QUALITY_REPORT_ROOT / 'metadata_issues.csv')
validated_manifest_df = pd.read_parquet(QUALITY_MANIFEST_ROOT / QUALITY_MANIFEST_NAME)
metadata_decision = metadata_summary['decision']

Validação de qualidade reutilizada: minds-libras-raw


## 8.1 Evidências da decisão

O resumo JSON e o CSV de issues são a fonte de evidência da decisão. O Parquet é um manifest derivado, com metadados observados e flags de uso; nenhum arquivo em `data/raw/` é sobrescrito.

In [47]:
display(pd.DataFrame([{'dataset': metadata_summary['dataset'], 'decision': metadata_decision, **metadata_summary['counts']['by_severity']}]))
display(metadata_issues_df.head(50))
display(validated_manifest_df.head())

,dataset,decision,CRITICAL,ERROR,WARNING,INFO
0,minds-libras-raw,GO,0,0,0,1


,dataset,severity,code,video_name,source,row_number,message,evidence
0,minds-libras-raw,INFO,error_csv_absent,NaN,error.csv,NaN,dataset não possui error.csv; nenhuma exceção ...,NaN


,video_id,video_name,class,user_id,width,height,fps,dataset,filesystem_key_column,filesystem_key,...,actual_duration,codec,file_exists,file_readable,filename_valid,descriptive_name_status,descriptive_name_note,metadata_valid,validation_status,validation_notes
0,01AcontecerSinalizador01-1.mp4,Acontecer_Sinalizador01-1.mp4,Acontecer,Sinalizador01,1920,1080,104.0,minds-libras-raw,video_id,01AcontecerSinalizador01-1.mp4,...,3.434,h264,True,True,None,deterministic_prefix_and_separator,video_id adiciona prefixo numérico e remove o ...,True,valid,chave física MINDS validada por video_id
1,01AcontecerSinalizador01-2.mp4,Acontecer_Sinalizador01-2.mp4,Acontecer,Sinalizador01,1920,1080,130.0,minds-libras-raw,video_id,01AcontecerSinalizador01-2.mp4,...,4.300,h264,True,True,None,deterministic_prefix_and_separator,video_id adiciona prefixo numérico e remove o ...,True,valid,chave física MINDS validada por video_id
2,01AcontecerSinalizador01-3.mp4,Acontecer_Sinalizador01-3.mp4,Acontecer,Sinalizador01,1920,1080,120.0,minds-libras-raw,video_id,01AcontecerSinalizador01-3.mp4,...,3.967,h264,True,True,None,deterministic_prefix_and_separator,video_id adiciona prefixo numérico e remove o ...,True,valid,chave física MINDS validada por video_id
3,01AcontecerSinalizador01-4.mp4,Acontecer_Sinalizador01-4.mp4,Acontecer,Sinalizador01,1920,1080,121.0,minds-libras-raw,video_id,01AcontecerSinalizador01-4.mp4,...,4.000,h264,True,True,None,deterministic_prefix_and_separator,video_id adiciona prefixo numérico e remove o ...,True,valid,chave física MINDS validada por video_id
4,01AcontecerSinalizador01-5.mp4,Acontecer_Sinalizador01-5.mp4,Acontecer,Sinalizador01,1920,1080,131.0,minds-libras-raw,video_id,01AcontecerSinalizador01-5.mp4,...,4.334,h264,True,True,None,deterministic_prefix_and_separator,video_id adiciona prefixo numérico e remove o ...,True,valid,chave física MINDS validada por video_id


## 8.2 Como interpretar o resultado

Se `status_geral`, `status_do_esquema` e `status_da_validação` estiverem como `ok`, o arquivo foi lido e os campos encontrados estão no formato básico esperado.

Isso ainda não garante que o vídeo seja válido para treinamento. Aqui verificamos se os metadados podem ser lidos e se os registros não apresentam conflitos básicos de identificação. Regras como dimensões positivas, FPS plausível, vídeo corrompido e distribuição das classes ficam para etapas posteriores.

# 9. Artefatos e relatório da execução

Esta seção cria um resumo humano da execução em `harness/artifacts/reports/executions/`. Os relatórios técnicos completos continuam em `app/data/processed/metadata-validation/`.

A atualização da arquitetura só é chamada se `RUN_ARCHITECTURE_UPDATE = True` e existe um candidato Archify em `harness/artifacts/architecture/source/`.

In [48]:
HARNESS_ROOT = PROJECT_ROOT.parent / 'harness'
ARTIFACTS_ROOT = HARNESS_ROOT / 'artifacts'
EXECUTION_REPORTS_ROOT = ARTIFACTS_ROOT / 'reports' / 'executions'
ARCHITECTURE_ROOT = ARTIFACTS_ROOT / 'architecture'

def _report_counts(values: dict) -> str:
    if not values:
        return 'nenhuma contagem disponível'
    return ', '.join(f'{key}={value}' for key, value in values.items())

if RUN_EXECUTION_REPORT:
    EXECUTION_REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
    report_time = datetime.now().astimezone()
    report_id = report_time.strftime('%Y%m%d_%H%M%S')
    execution_report_path = EXECUTION_REPORTS_ROOT / f'{METADATA_DATASET}_{report_id}.md'
    technical_summary = metadata_validation.get('summary', {})
    quality_counts = metadata_summary.get('counts', {}).get('by_severity', {})
    fps_table = fps_hypotheses_summary_df.to_string(index=False)
    report_lines = [
        f'# Relatório da execução — {METADATA_DATASET}',
        '',
        f'- Executado em: {report_time.isoformat()}',
        f'- Dataset: `{METADATA_DATASET}`',
        f'- Decisão da validação de qualidade: **{metadata_decision}**',
        '',
        '## Controle utilizado',
        '',
        f'- Aquisição habilitada: `{RUN_DATASET_ACQUISITION}`',
        f'- Download forçado: `{FORCE_DATASET_DOWNLOAD}`',
        f'- Build Docker forçado: `{FORCE_DOCKER_BUILD}`',
        f'- Validação técnica habilitada: `{RUN_METADATA_VALIDATION}`',
        f'- Validação de qualidade habilitada: `{RUN_QUALITY_VALIDATION}`',
        f'- Reexecução forçada: `{FORCE_RERUN}`',
        '',
        '## Resultado técnico',
        '',
        f'- Resumo: `{_report_counts(technical_summary)}`',
        '- Hipóteses de `fps`:',
        '```text',
        fps_table,
        '```',
        '',
        '## Issues e proveniência',
        '',
        f'- Severidades: `{_report_counts(quality_counts)}`',
        f'- Relatórios técnicos: `{QUALITY_REPORT_ROOT}`',
        f'- Manifest validado: `{QUALITY_MANIFEST_ROOT / QUALITY_MANIFEST_NAME}`',
        '',
        '## Observação',
        '',
        'Este resumo é derivado. Os arquivos em `app/data/raw/` permanecem fora do fluxo de escrita.',
    ]
    execution_report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
    latest_report_path = ARTIFACTS_ROOT / 'reports' / 'latest.md'
    latest_report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
    latest_summary = {
        'schema_version': 'execution-report.v1',
        'dataset': METADATA_DATASET,
        'executed_at': report_time.isoformat(),
        'decision': metadata_decision,
        'technical_summary': technical_summary,
        'quality_counts': quality_counts,
        'report': str(execution_report_path),
    }
    (ARTIFACTS_ROOT / 'reports' / 'latest.json').write_text(
        json.dumps(latest_summary, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print(f'Relatório gerado: {execution_report_path}')
    try:
        from IPython.display import Markdown, display
        display(Markdown('\n'.join(report_lines)))
    except ImportError:
        pass
else:
    print('Relatório da execução desabilitado por RUN_EXECUTION_REPORT.')

if RUN_ARCHITECTURE_UPDATE:
    refresh_script = HARNESS_ROOT / 'scripts' / 'refresh_architecture.ps1'
    if not refresh_script.is_file():
        raise FileNotFoundError(f'Hook de arquitetura não encontrado: {refresh_script}')
    refresh = subprocess.run(
        ['powershell.exe', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(refresh_script), '-Dataset', METADATA_DATASET],
        cwd=HARNESS_ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace', check=False,
    )
    if refresh.stdout:
        print(refresh.stdout)
    if refresh.returncode != 0:
        if refresh.stderr:
            print(refresh.stderr)
        raise RuntimeError(f'Atualização da arquitetura falhou com código {refresh.returncode}.')
    print(f'Estado da arquitetura: {ARCHITECTURE_ROOT / "architecture-refresh.json"}')
else:
    print('Atualização da arquitetura desabilitada por RUN_ARCHITECTURE_UPDATE.')

Relatório gerado: c:\Users\mdbaa\development\Dissertacao\harness\artifacts\reports\executions\minds-libras-raw_20261025_102244.md


# Relatório da execução — minds-libras-raw

- Executado em: 2026-10-25T10:22:44.647529-03:00
- Dataset: `minds-libras-raw`
- Decisão da validação de qualidade: **GO**

## Controle utilizado

- Aquisição habilitada: `True`
- Download forçado: `False`
- Build Docker forçado: `False`
- Validação técnica habilitada: `True`
- Validação de qualidade habilitada: `True`
- Reexecução forçada: `False`

## Resultado técnico

- Resumo: `dataset=minds-libras-raw, videos_seen=800, videos_ok=800, videos_partial=0, videos_with_probe_error=0, width_matches=800, height_matches=800, fps_rate_matches=0, fps_frame_count_matches=800, fps_none_matches=0, fps_inconclusive=0, fps_hypothesis_counts={'frame_count': 800}, dataset_fps_interpretation=frame_count, fps_tolerance=0.01`
- Hipóteses de `fps`:
```text
 hypothesis  count
frame_count    800
```

## Issues e proveniência

- Severidades: `CRITICAL=0, ERROR=0, WARNING=0, INFO=1`
- Relatórios técnicos: `c:\Users\mdbaa\development\Dissertacao\app\data\processed\metadata-validation\minds-libras-raw\reports`
- Manifest validado: `c:\Users\mdbaa\development\Dissertacao\app\data\processed\metadata-validation\minds-libras-raw\manifests\minds_libras_validated.parquet`

## Observação

Este resumo é derivado. Os arquivos em `app/data/raw/` permanecem fora do fluxo de escrita.

Estado da arquitetura: c:\Users\mdbaa\development\Dissertacao\harness\artifacts\architecture\architecture-refresh.json
